In [1]:
import sys
PATH_TO_PREDICTOR_HOME = ".."
sys.path.append(PATH_TO_PREDICTOR_HOME)


from mhcnames import normalize_allele_name
import numpy as np
import pandas as pd
from config import RESULT_COLUMNS

# Params

In [2]:
CLASSES_TO_GENERATE = ['I','II']
ORGANISMS_TO_GENERATE = ['Human', 'Mouse']

# 0.1 Read source decoy data file

In [3]:
DECOY_DATA_PATH = "G:/My Drive/Work/ITMO/data/decoy/"

In [4]:
per_organism_per_class_decoy_files = {
    organism: {
        class_to_generate:  pd.read_csv("{}decoy-{}-class-{}.txt".format(DECOY_DATA_PATH, organism.lower(),class_to_generate.lower()), header=None, names = ["peptide"])
            for class_to_generate in CLASSES_TO_GENERATE
        }
      for organism in ORGANISMS_TO_GENERATE
    }

## Filter canonical AAs

In [5]:
def filter_canonical_aa(df):
    df = df[~df['peptide'].str.contains("U")]
    df = df[~df['peptide'].str.contains("\+")]
    df = df[~df['peptide'].str.contains("X")]
    df = df[~df['peptide'].str.contains("B")]
    df = df[~df['peptide'].str.contains("Z")]
    df = df[~df['peptide'].str.contains("O")]
    df = df[~df['peptide'].str.contains("J")]
    df = df[~df['peptide'].str.contains("-")]
    return df

In [6]:
for organism in ORGANISMS_TO_GENERATE:
    for class_to_generate in CLASSES_TO_GENERATE:
        df = per_organism_per_class_decoy_files[organism][class_to_generate]
        df = filter_canonical_aa(df.drop_duplicates())
        print("Remaining length is {}".format(len(df)))
        per_organism_per_class_decoy_files[organism][class_to_generate] = df

Remaining length is 10170680
Remaining length is 6555557
Remaining length is 7069953
Remaining length is 4356057


# 0.2 Sample peptides

In [7]:
# sample size for each length
SAMPLE_SIZE = 200_000
for organism in ORGANISMS_TO_GENERATE:
    for class_to_generate in CLASSES_TO_GENERATE:
        df = per_organism_per_class_decoy_files[organism][class_to_generate]
        df['length'] =df.peptide.str.len()
        sampled_df = df.groupby('length').sample(SAMPLE_SIZE)
        per_organism_per_class_decoy_files[organism][class_to_generate] = sampled_df



# 0.3 Save to file

In [8]:
for organism in ORGANISMS_TO_GENERATE:
    for class_to_generate in CLASSES_TO_GENERATE:
        df = per_organism_per_class_decoy_files[organism][class_to_generate]
        df.to_csv("{}sampled_decoy_data_{}_class_{}.csv".format(DECOY_DATA_PATH,organism.lower(),class_to_generate.lower()), index=False)

In [9]:
df.peptide.str.len().unique()

array([11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27],
      dtype=int64)